# Step 1b: Audio → Text (Whisper) — Google Colab version

Standalone Colab clone of the Whisper cells from `01_extract_raw_text.ipynb`.
Runs on Colab's free GPU (Runtime → Change runtime type → GPU) so we can use
`medium` instead of `small` (local machine has no CUDA).

Uses the `hsk-paper-exam` folder already uploaded to your Google Drive (135 files,
PDFs + audio mixed) — the inventory step below filters to audio only.

Output (`raw_extractions_audio.parquet/csv`) is saved back to Drive — copy it into
`data/raw/` locally afterwards and merge with `df_pdf` as in the original notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q openai-whisper pyarrow

In [ ]:
import re, warnings
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import whisper
warnings.filterwarnings("ignore")

# Your Drive folder (contains both PDFs and audio -- inventory below filters to audio only)
DATA_DIR   = Path("/content/drive/MyDrive/hsk-paper-exam")
OUTPUT_DIR = Path("/content/drive/MyDrive/hsk-freq-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_parquet(df, path):
    # cast large_string -> string so local parquet viewers (e.g. VS Code) can read it
    table = pa.Table.from_pandas(df, preserve_index=False)
    fields = [f.with_type(pa.string()) if pa.types.is_large_string(f.type) else f
              for f in table.schema]
    table = table.cast(pa.schema(fields))
    pq.write_table(table, path)

print("CUDA available:", torch.cuda.is_available())
print("Data dir:", DATA_DIR)

## 1 — Parse filename + inventory (audio only)

In [ ]:
EXAM_ID_RE = re.compile(r"^(H([34])\d{4}[A-Z]?)", re.IGNORECASE)

def parse_filename(filename):
    m = EXAM_ID_RE.match(filename)
    if not m: return None, None
    return m.group(1).upper(), int(m.group(2))

def file_priority(filename):
    name = filename.lower()
    score = 0
    if "_copy"  in name: score += 10
    if "-audio" in name: score += 10
    if "试卷"   in name: score += 5
    if "(1)"    in name: score += 3
    return score

AUDIO_EXTS = (".mp3", ".wav", ".m4a", ".flac")

rows = []
for f in sorted(DATA_DIR.iterdir()):
    if f.suffix.lower() not in AUDIO_EXTS: continue
    exam_id, hsk_level = parse_filename(f.name)
    if not exam_id:
        print(f"[SKIP] {f.name}"); continue
    rows.append({"exam_id": exam_id, "hsk_level": hsk_level,
        "source_type": "listening", "filename": f.name, "filepath": str(f),
        "size_mb": round(f.stat().st_size / 1_048_576, 2),
        "_priority": file_priority(f.name)})

raw_inv = pd.DataFrame(rows)
inventory = (
    raw_inv.sort_values("_priority")
    .drop_duplicates(subset=["exam_id", "source_type"], keep="first")
    .drop(columns=["_priority"])
    .sort_values(["hsk_level", "exam_id"])
    .reset_index(drop=True)
)

dups = raw_inv[raw_inv.duplicated(subset=["exam_id", "source_type"], keep=False)]
if len(dups):
    print("Duplicates removed:")
    for (eid, st), g in dups.groupby(["exam_id", "source_type"]):
        g = g.sort_values("_priority") if "_priority" in g else g
        print(f"  {eid} [{st}] keep={g.iloc[0]['filename']}")

print(f"raw={len(raw_inv)}  after dedup={len(inventory)}")
inventory

## 2 — Load Whisper (GPU)

In [ ]:
WHISPER_MODEL = "medium"  # tiny | base | small | medium | large -- Colab GPU can handle medium/large
TEST_ONLY     = True       # flip to False once a 1-file test looks right

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper [{WHISPER_MODEL}] on {device} ...")
model = whisper.load_model(WHISPER_MODEL, device=device)
print("Whisper ready.")

## 3 — Transcribe (checkpointed to Drive so a disconnect doesn't lose progress)

In [ ]:
audio_rows = inventory.copy()
if TEST_ONLY:
    audio_rows = audio_rows.head(1)
    print(f"TEST_ONLY -- transcribing: {audio_rows.iloc[0]['filename']}")

AUDIO_CKPT = OUTPUT_DIR / "_audio_progress.parquet"

if AUDIO_CKPT.exists():
    done_df = pd.read_parquet(AUDIO_CKPT)
    done_filenames = set(done_df["filename"])
    audio_results = done_df.to_dict("records")
    print(f"Resuming -- {len(done_filenames)} files already done, skipping them")
else:
    done_filenames = set()
    audio_results = []

remaining = audio_rows[~audio_rows["filename"].isin(done_filenames)]

for i, (_, row) in enumerate(remaining.iterrows(), 1):
    print(f"[{i}/{len(remaining)}] {row['filename']} ({row['size_mb']} MB) ...")
    try:
        result = model.transcribe(row["filepath"], language="zh", temperature=0.0)
        text, note = result["text"], "ok"
    except Exception as e:
        text, note = "", f"error:{e}"
    audio_results.append({**row.to_dict(), "text": text, "text_length": len(text), "notes": note})
    print(f"   {len(text)} chars | {repr(text[:80].strip())}")
    save_parquet(pd.DataFrame(audio_results).drop(columns=["filepath"], errors="ignore"), AUDIO_CKPT)

df_audio = pd.DataFrame(audio_results).drop(columns=["filepath"], errors="ignore")
print(f"\nDone. {len(df_audio)} audio files")
df_audio.head()

## 4 — Save final output to Drive

Copy the resulting files from `hsk-freq-output/` in your Drive into the local
`data/raw/` folder, then `pd.concat` with `df_pdf` as in the original notebook's cell 19.

In [ ]:
df_audio.to_csv(OUTPUT_DIR / "raw_extractions_audio.csv", index=False, encoding="utf-8-sig")
save_parquet(df_audio, OUTPUT_DIR / "raw_extractions_audio.parquet")
print(f"Saved -> {OUTPUT_DIR / 'raw_extractions_audio.csv/.parquet'}")